# 評估 Agents

我們已經有一個 email assistant，它用 router 來對 email 進行 triage，然後把 email 交給 agent 產生回應。但我們怎麼能確定它在正式環境（production）中會運作良好呢？這正是測試之所以重要的原因：它能用可量化的指標（例如回應品質、token 用量、延遲，或 triage 準確率）來引導我們對 agent 架構的決策。[LangSmith](https://docs.smith.langchain.com/) 提供了兩種主要的方式來測試 agent。

![overview-img](img/overview_eval.png)

#### 載入環境變數

In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

## 如何執行 Evaluation

#### Pytest / Vitest

[Pytest](https://docs.pytest.org/en/stable/) 與 Vitest 對許多開發者來說都很熟悉，它們是 Python 與 JavaScript 生態系中強大的測試工具。LangSmith 與這些框架整合，讓你能撰寫並執行測試，並把結果記錄到 LangSmith。在這份 notebook 中，我們會使用 Pytest。
* 對於已經熟悉自己框架的開發者來說，Pytest 是很好的入門方式。
* Pytest 也很適合用在較複雜的 evaluation：當每個 agent 測試案例都需要特定的檢查與成功條件、而這些較難一般化時。

#### LangSmith Datasets

你也可以[在 LangSmith 中](https://docs.smith.langchain.com/evaluation)建立一個 dataset，並使用 LangSmith 的 evaluate API 讓我們的 assistant 對該 dataset 進行測試。
* 對於協作建立測試套件的團隊來說，LangSmith datasets 非常適合。
* 你可以善用正式環境的 traces、annotation queue、合成資料生成等方式，持續為一個不斷成長的 golden dataset 加入範例。
* 當你能定義出可套用到 dataset 中每個測試案例的 evaluator（例如相似度、完全比對的準確率等）時，LangSmith datasets 就非常合適。

## Test Cases

測試通常會從定義測試案例開始，而這可能是個很有挑戰性的過程。在這個情況下，我們只會定義一組想要處理的範例 email，以及幾項想要測試的項目。你可以在 `eval/email_dataset.py` 中看到這些測試案例，其中包含以下內容：

1. **Input Emails**：一組多樣化的 email 範例
2. **Ground Truth Classifications**：`Respond`、`Notify`、`Ignore`
3. **Expected Tool Calls**：針對每封需要回應的 email，預期會被呼叫的 tools
4. **Response Criteria**：對於需要回覆的 email，什麼樣才算是好的回應

請注意，我們同時具備：
- 端到端的「整合（integration）」測試（例如：Input Emails -> Agent -> Final Output，再與 Response Criteria 比對）
- 針對 workflow 中特定步驟的測試（例如：Input Emails -> Agent -> Classification，再與 Ground Truth Classification 比對）

In [2]:

%load_ext autoreload
%autoreload 2

from email_assistant.eval.email_dataset import email_inputs, expected_tool_calls, triage_outputs_list, response_criteria_list

# 本章的前提，這個沒講清楚，後面全是天書：
# 傳統單元測試靠 assert result == expected，因為 f(x) 每次都吐一模一樣的東西。
# 但 LLM 每次回答都不一樣——同一封信跑兩次，措辭不會相同。硬比對字串必然失敗。
#
# 重點不是「LLM 都沒有標準答案」，而是「有些事有、有些事沒有」。
# 下面這四種測試資料，正好對應三種評估方法，分工看這裡：
#
#   triage_outputs_list（該分成 ignore / notify / respond）
#     → 有標準答案（三選一）。用 LangSmith Dataset + evaluator，跑整份題庫看統計。
#   expected_tool_calls（該呼叫 write_email / check_calendar_availability）
#     → 有標準答案（tool 名字是固定字串）。用 Pytest 硬比對。
#   response_criteria_list（回信寫得好不好）
#     → 沒有標準答案。「好的回信」沒有唯一寫法，只好請另一個 LLM 依 criteria 評分。
#
# ⚠️ 判準記這一句就好：有標準答案就硬比對，沒有才動用 LLM 當裁判。
#    LLM 裁判又貴又慢，而且裁判自己也會判錯——能不用就不用。
test_case_ix = 0

print("Email Input:", email_inputs[test_case_ix])
print("Expected Triage Output:", triage_outputs_list[test_case_ix])
print("Expected Tool Calls:", expected_tool_calls[test_case_ix])
print("Response Criteria:", response_criteria_list[test_case_ix])

Email Input: {'author': 'Alice Smith <alice.smith@company.com>', 'to': 'Lance Martin <lance@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': "Hi Lance,\n\nI was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?\n\nSpecifically, I'm looking at:\n- /auth/refresh\n- /auth/validate\n\nThanks!\nAlice"}
Expected Triage Output: respond
Expected Tool Calls: ['write_email', 'done']
Response Criteria: 
• Send email with write_email tool call to acknowledge the question and confirm it will be investigated  



### 一張圖看懂：四種測試資料 → 三條評估路線

上面印出來的四樣東西不是隨便湊的，它們**各自餵給不同的評估方法**。整份 notebook 剩下的部分就是在跑這三條線：

```
                        email_inputs（16 封測試信）
                                   │
         ┌─────────────────────────┼─────────────────────────┐
         │                         │                         │
  triage_outputs_list      expected_tool_calls       response_criteria_list
 「該 ignore / notify      「該呼叫哪些 tool」        「回信要滿足什麼」
   / respond？」                                     • 要回答缺的 endpoint
   → "respond"             → ["write_email",         • 語氣專業
                              "done"]                • 不可捏造
         │                         │                         │
    ✅ 有標準答案             ✅ 有標準答案              ❌ 沒有標準答案
    （三選一）                （tool 名是固定字串）      （好的回信沒有唯一寫法）
         │                         │                         │
         ▼                         ▼                         ▼
 ┌───────────────────┐   ┌───────────────────┐   ┌───────────────────────┐
 │ ② LangSmith       │   │ ① Pytest          │   │ ③ LLM-as-judge        │
 │    Dataset        │   │    硬比對          │   │                       │
 ├───────────────────┤   ├───────────────────┤   ├───────────────────────┤
 │ client.evaluate(  │   │ missing = [...]   │   │ judge.invoke(         │
 │   target,         │   │ assert not        │   │   criteria +          │
 │   data,           │   │        missing    │   │   response)           │
 │   evaluators)     │   │                   │   │                       │
 ├───────────────────┤   ├───────────────────┤   ├───────────────────────┤
 │ → 統計            │   │ → 綠燈 / 紅燈     │   │ → True / False        │
 │   「85% 準確率」   │   │   「2 passed」    │   │   + justification     │
 ├───────────────────┤   ├───────────────────┤   ├───────────────────────┤
 │ 跑整份題庫、       │   │ 快、幾乎免費、     │   │ 貴、慢、              │
 │ 看跨版本趨勢       │   │ 適合放進 CI       │   │ 而且裁判自己會判錯     │
 └───────────────────┘   └───────────────────┘   └───────────────────────┘
      本 notebook            本 notebook              本 notebook
      「LangSmith            「Pytest 範例」           「LLM-as-Judge
        Datasets 範例」        那一段                    Evaluation」那一段
```

> ⚠️ **判準只有一句話：有標準答案就硬比對，沒有才動用 LLM 當裁判。**
>
> 很多人一上來就用 LLM-as-judge 評所有東西，因為它聽起來最智慧。但它**貴**（每個 case 多一次 LLM 呼叫）、**慢**（CI 從 30 秒變 10 分鐘）、而且**它自己也會錯**——你等於用一個不確定的東西去測另一個不確定的東西。能用 `==` 解決的，絕不叫 LLM。

## Pytest 範例

我們來看看如何用 Pytest 為 workflow 中的某個特定部分撰寫測試。我們將測試當 `email_assistant` 在回應 email 時，是否做出了正確的 tool calls。

### 先搞懂：pytest 到底在幹嘛？

如果你沒寫過傳統軟體測試，pytest 會像黑魔法：**你從頭到尾沒有呼叫任何函式，打一個指令，它自己就跑起來了。** 先把這件事講清楚，下面的程式碼才不會像天書。

#### 核心魔法：它自己去找測試

平常寫 Python，你得自己呼叫：

```python
result = add(2, 3)
if result != 5:
    print("錯了")
```

用 pytest，你只要寫一個**「長得像測試」的函式**，然後什麼都不做：

```python
def test_add():
    assert add(2, 3) == 5
```

接著在終端機打 `pytest`，它會自己：

1. 掃描資料夾，找出所有檔名符合 `test_*.py` 的檔案
2. 在裡面找出所有 `test_` 開頭的函式
3. 一個一個執行
4. **沒拋出例外 = 通過（綠燈）；`assert` 失敗 = 沒過（紅燈）**

**全部靠命名慣例。** 檔案不叫 `test_` 開頭 → 它看不到。函式不叫 `test_` 開頭 → 它不會跑。

> ⚠️ **最常見的困惑**：「我明明寫了測試，為什麼 pytest 說 collected 0 items？」九成是名字沒取對。這也是為什麼這個專案的檔案叫 `test_response.py`、函式叫 `test_email_dataset_tool_calls`——**那不是命名品味，是 pytest 的尋人啟事**。

#### `assert` 就是全部

pytest 沒有 `assertEqual()` / `assertTrue()` 那一套（那是 unittest）。你就寫最普通的 Python `assert`：

```python
assert x == 5
assert "abc" in text
assert not missing_calls
```

pytest 會在背後**改寫**你的 assert，所以失敗時它能告訴你「x 實際上是 7」，而不是只丟一句 `AssertionError`。

#### 那幾個 `@` 是什麼

| 寫法 | 它做什麼 |
|---|---|
| `@pytest.mark.parametrize("a,b", [(1,2),(3,4)])` | **一個函式跑多組資料**，每組算一個獨立測試。2 組資料 = 報告上 2 筆，一筆紅另一筆照樣綠。這是純 pytest 功能。 |
| `@pytest.mark.langsmith` | **LangSmith 外掛**加的，讓這個測試的結果上傳成一次實驗。不是 pytest 內建。 |
| `@pytest.fixture` | 準備測試要用的東西（例如共用的設定）。這個專案放在 `tests/conftest.py`——**那個檔名也是慣例，pytest 會自動讀它，不用 import**。 |

#### 指令怎麼讀

```bash
pytest tests/test_response.py --agent-module email_assistant
   │        │                        └─ 這個專案自訂的參數，定義在 tests/conftest.py，
   │        │                           用來切換要測哪一版 assistant（不是 pytest 內建）
   │        └─ 只跑這個檔案（不給的話它會掃整個資料夾）
   └─ 指令本身
```

幾個好用的旗標：

| 旗標 | 意思 |
|---|---|
| `--collect-only` | **只列出「會跑哪些測試」，不真的跑**。想確認有沒有被找到、又不想燒錢時最好用 |
| `-x` | 第一個失敗就停，別浪費時間 |
| `-q` | 安靜模式，輸出少一點 |
| `-m "not llm_judge"` | 只跑**沒有**被標記成 `llm_judge` 的測試（用來把貴的測試隔開） |

#### 為什麼不自己寫一堆 if/else 就好？

你當然可以。pytest 換來的是四件事：

- **自動發現**：不用手動維護一份「所有測試」的清單，加一個 `test_` 函式它就跑得到。
- **彼此隔離**：一個測試炸掉，其他照跑，不會整組停擺。
- **失敗看得懂**：它告訴你哪個檔案哪一行、期待什麼、實際什麼。
- **生態系**：`parametrize` 讓你一份程式碼跑 16 個 case，LangSmith 外掛讓結果自動上傳——這正是下面要做的事。

**一句話**：pytest 是一個「照名字自動找出你的測試、一個一個跑、然後告訴你哪個壞了」的工人。你的工作只有兩件——**把函式取名 `test_` 開頭，然後寫 `assert`**。

In [3]:
import pytest
from email_assistant.eval.email_dataset import email_inputs, expected_tool_calls
from email_assistant.utils import format_messages_string
from email_assistant.email_assistant import email_assistant
from email_assistant.utils import extract_tool_calls

from langsmith import testing as t

# 這兩個 decorator 來自不同世界，別混為一談：
#
# @pytest.mark.parametrize → 純 pytest 功能，跟 LangSmith 一點關係都沒有。
#   一個測試函式跑多組資料，每組算「一筆獨立的測試」。
#   這裡給 2 組 = 2 個測試、2 列結果，不是「一個測試檢查兩封信」。
#   差別很實際：一封過、一封掛的時候，你分得出來是哪封掛。
#
# @pytest.mark.langsmith → 這個才是 LangSmith 的整合點。
#   加了它，測試結果會自動上傳成一次實驗（在 UI 的 Datasets & Experiments 底下）。
#   拿掉它測試照跑照過，只是什麼紀錄都不留。
@pytest.mark.langsmith
@pytest.mark.parametrize(
    "email_input, expected_calls",
    [   # 挑幾個預期會回信的範例
        (email_inputs[0],expected_tool_calls[0]),
        (email_inputs[3],expected_tool_calls[3]),
    ],
)
def test_email_dataset_tool_calls(email_input, expected_calls):
    """測試 email 處理流程是否包含預期的 tool calls。
    
    這個測試確認所有預期的 tools 都在 email 處理過程中被呼叫，
    但不檢查 tool 呼叫的順序，也不檢查每個 tool 被呼叫的次數。
    如有需要，可以額外加入針對這些面向的檢查。
    """
    # 執行 email assistant
    #
    # ⚠️ 踩雷：下面這兩行是錯的，真的跑下去會 KeyError: 'email_input'。
    #    email_assistant 編譯時寫的是 StateGraph(State, input=StateInput)，
    #    而 StateInput 只認 email_input 這一個 key。傳 messages 進去會被 input schema 濾掉，
    #    triage_router 第一行取 state["email_input"] 就炸。
    #    正確寫法在 notebooks/test_tools.py（markdown 說「上面那段程式碼」指的就是它）：
    #        result = email_assistant.invoke({"email_input": email_input})
    #    這個 cell 只「定義」測試函式、不會真的執行測試，所以在 notebook 裡錯誤不會浮現。
    #    上台若要現場跑這個測試，請用 test_tools.py 那個版本。
    messages = [{"role": "user", "content": str(email_input)}]
    result = email_assistant.invoke({"messages": messages})
            
    # 從 messages 清單中取出 tool calls
    extracted_tool_calls = extract_tool_calls(result['messages'])
            
    # ⚠️ 注意這個檢查有多寬鬆：它只問「預期的 tool 有沒有出現過」。
    #    不管順序、不管次數、也不管它有沒有多呼叫別的 tool（docstring 有寫，但值得講出來）。
    #    寬鬆是刻意的取捨——LLM 先查行事曆再回信、或先回信再補查，都算合理，
    #    綁死順序等於逼測試去測「本來就不該固定的東西」，只會得到一堆假失敗。
    #    但代價要講清楚：這個測試抓不到「多呼叫了不該呼叫的 tool」。
    #    assistant 就算多寄了三封莫名其妙的信，只要該呼叫的那個有出現，照樣綠燈。
    missing_calls = [call for call in expected_calls if call.lower() not in extracted_tool_calls]
    
    # 為什麼不直接 assert 就好，還要多這一段？
    # assert 失敗只告訴你「錯了」，紅燈一亮，你兩手空空。
    # log_outputs 把「LLM 這次實際做了什麼」留在 LangSmith 上，失敗時才有現場可查。
    # 這是評估 LLM 系統跟測傳統程式最大的差別：
    # 傳統程式掛了，看 stack trace 就知道哪行爆；LLM 判錯了，stack trace 什麼也沒有，
    # 你得回頭看它到底講了什麼、呼叫了哪些 tool，才知道是 prompt 爛、還是題目本身有歧義。
    # 一句話：你需要看現場，不只看紅綠燈。
    t.log_outputs({
                "missing_calls": missing_calls,
                "extracted_tool_calls": extracted_tool_calls,
                "response": format_messages_string(result['messages'])
            })

    # 若沒有任何預期的呼叫缺漏，測試即通過
    assert len(missing_calls) == 0

In [ ]:
# ⬆️ 上面那一格只是「定義」了一個測試函式——它從頭到尾沒有被執行過。
# 整個 Pytest 段落到這裡為止，你其實還沒看過 pytest 跑起來長什麼樣。
# 這一格就是讓你親眼看到。
#
# 跑的是 notebooks/test_tools.py，內容和上面那格幾乎一樣（教材的 markdown 也叫你參考它）。
#
# ⚠️ 但要小心：上面那格用的是 invoke({"messages": messages})，實測會噴 KeyError: 'email_input'
#    ——因為 email_assistant 編譯時綁了 input schema（StateGraph(State, input=StateInput)），
#    StateInput 只收 email_input，messages 會被濾掉。test_tools.py 用的才是正確的
#    invoke({"email_input": email_input})。同一份教材，展示的版本是壞的、參考的檔案是對的。
#
# ⚠️ test_tools.py 原本還有一個 401 陷阱（已修好，可以打開來看註解）：
#    load_dotenv 必須放在 import email_assistant **之前**。因為 email_assistant.py 在
#    module 層級就 init_chat_model()，import 的瞬間 client 就把當時的 API key 複製走了，
#    之後再 load_dotenv 也來不及。這個坑的症狀是 401，錯誤訊息完全不會提到 .env。
#
# 執行時間約 20 秒（2 個 case，各跑一次真的 agent）。看輸出的三個重點：
#   collected 2 items  → pytest 靠檔名 test_*.py + 函式名 test_* 自己找到它們
#   [email_input0-...] → parametrize 把 2 組資料變成 2 個獨立測試
#   2 passed           → assert 沒炸 = 綠燈
!cd .. && LANGSMITH_TEST_SUITE='Email assistant: Tool Calls' pytest notebooks/test_tools.py -v

你會注意到幾件事。
- 要[用 Pytest 執行並把測試結果記錄到 LangSmith](https://docs.smith.langchain.com/evaluation/how_to_guides/pytest)，我們只需要在函式上加上 `@pytest.mark.langsmith` decorator，並把它放進一個檔案中，就像你在 `notebooks/test_tools.py` 中看到的那樣。這樣就會把測試結果記錄到 LangSmith。
- 其次，我們可以像[這裡](https://docs.smith.langchain.com/evaluation/how_to_guides/pytest#parametrize-with-pytestmarkparametrize)所示，透過 `@pytest.mark.parametrize` 把 dataset 範例傳入測試函式。

#### 執行 Pytest
我們可以從命令列執行這個測試。我們已把上面那段程式碼寫進一個 python 檔案中。從專案根目錄執行：

`! LANGSMITH_TEST_SUITE='Email assistant: Test Tools For Interrupt'  pytest notebooks/test_tools.py`

#### 檢視實驗結果

我們可以在 LangSmith UI 中檢視結果。`assert len(missing_calls) == 0` 會被記錄到 LangSmith 的 `Pass` 欄位。`log_outputs` 會被放進 `Outputs` 欄位，而函式參數則會被放進 `Inputs` 欄位。傳入 `@pytest.mark.parametrize(` 的每一筆輸入都會成為獨立的一列，記錄到 LangSmith 中名為 `LANGSMITH_TEST_SUITE` 的專案，你可以在 `Datasets & Experiments` 底下找到它。

![Test Results](img/test_result.png)

## LangSmith Datasets 範例

![overview-img](img/eval_detail.png)

我們來看看如何使用 LangSmith datasets 來執行 evaluation。在前一個 Pytest 範例中，我們評估了 email assistant 的 tool calling 準確率。而現在我們要在這裡評估的 dataset，則是專門針對 email assistant 的 triage 步驟，也就是分類一封 email 是否需要回應。

#### Dataset Definition

我們可以用 LangSmith SDK [在 LangSmith 中建立一個 dataset](https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets_programmatically#create-a-dataset)。下面這段程式碼會用 `eval/email_dataset.py` 檔案中的測試案例來建立一個 dataset。

In [4]:
from langsmith import Client

from email_assistant.eval.email_dataset import examples_triage

# 初始化 LangSmith client
client = Client()

# Dataset 名稱
dataset_name = "E-mail Triage Evaluation"

# Dataset 是存在 LangSmith 雲端的「題庫」，不是本地變數。
# 建一次就長在你的專案裡，之後每次實驗都對同一份題庫跑分，
# 才能回答「我這版改動到底有沒有比上一版好」——這才是評估的重點，不是跑一次看爽的。
#
# ⚠️ 踩雷：這個 if 只看「名字存不存在」，不看內容。
#    你之後改了 email_dataset.py 裡的測試案例、再重跑這個 cell，
#    因為同名 dataset 已經存在，整段直接跳過——雲端還是舊題庫，而且不會有任何錯誤訊息。
#    你會以為在測新案例，其實在測舊的。要更新就得換 dataset_name，或去 UI 砍掉重建。
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(
        dataset_name=dataset_name, 
        description="A dataset of e-mails and their triage decisions."
    )
    # 把範例加入 dataset
    client.create_examples(dataset_id=dataset.id, examples=examples_triage)

#### Target Function

這個 dataset 具有以下結構：以一封 email 作為輸入，並以該 email 的 ground truth triage 分類作為輸出：

```
examples_triage = [
  {
      "inputs": {"email_input": email_input_1},
      "outputs": {"classification": triage_output_1},   # NOTE: This becomes the reference_output in the created dataset
  }, ...
]
```

In [5]:
print("Dataset Example Input (inputs):", examples_triage[0]['inputs'])

Dataset Example Input (inputs): {'email_input': {'author': 'Alice Smith <alice.smith@company.com>', 'to': 'Lance Martin <lance@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': "Hi Lance,\n\nI was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?\n\nSpecifically, I'm looking at:\n- /auth/refresh\n- /auth/validate\n\nThanks!\nAlice"}}


In [6]:
print("Dataset Example Reference Output (reference_outputs):", examples_triage[0]['outputs'])

Dataset Example Reference Output (reference_outputs): {'classification': 'respond'}


我們定義一個函式，把 dataset 的輸入取出並傳給我們的 email assistant。LangSmith 的 [evaluate API](https://docs.smith.langchain.com/evaluation) 會把 `inputs` dict 傳給這個函式。接著這個函式會回傳一個帶有 agent 輸出的 dict。因為我們評估的是 triage 步驟，所以只需要回傳分類判斷的結果即可。

In [7]:
def target_email_assistant(inputs: dict) -> dict:
    """透過以 workflow 為基礎的 email assistant 處理一封 email。"""
    # 本章最實用的技巧：直接呼叫 graph 裡的「單一 node」，不跑整張圖。
    #
    # 平常的 email_assistant.invoke(...) 會從 START 一路跑到 END——
    # 分類完還會叫 agent 真的去查行事曆、寫回信。但這裡只想測「分類準不準」，
    # 後面寫不寫信根本無所謂。讓它跑完整張圖，等於每筆測試多燒好幾次 LLM 呼叫：
    # 慢、貴，而且會把「回信寫壞」的雜訊混進「分類測試」的結果裡，測不乾淨。
    # .nodes['triage_router'] 就是把那個 node 單獨拉出來跑。
    # 類比：只想驗收櫃檯分信分得對不對，沒必要每次都讓整間辦公室把信都回完。
    #
    # ⚠️ 踩雷：為什麼要 response.update['...']，不能直接 response['...']？
    #    因為 triage_router 回傳的是 Command 物件（第 2 章那個），不是普通的 state dict。
    #    Command 身上有兩個欄位：.update = 要寫進 state 的內容、.goto = 下一站去哪個 node。
    #    跑整張圖時 LangGraph 會幫你把 Command 拆開、把 .update 併進 state，你感覺不到它存在；
    #    但你單獨呼叫一個 node，就沒人幫你拆——拿到的是原封不動的 Command。
    #    直接寫 response['classification_decision'] 會炸。這是「只跑單一 node」的代價。
    response = email_assistant.nodes['triage_router'].invoke({"email_input": inputs["email_input"]})
    return {"classification_decision": response.update['classification_decision']}

#### Evaluator Function

現在，我們來建立一個 evaluator 函式。我們想評估什麼呢？我們的 dataset 中有 reference outputs，上面的函式中也有 agent 的輸出。

* Reference outputs：`"reference_outputs": {"classification": triage_output_1} ...`
* Agent outputs：`"outputs": {"classification_decision": agent_output_1} ...`

我們想評估 agent 的輸出是否與 reference output 相符。所以我們只需要一個 evaluator 函式來比較這兩者，其中 `outputs` 是 agent 的輸出，而 `reference_outputs` 是來自 dataset 的 reference output。

In [8]:
def classification_evaluator(outputs: dict, reference_outputs: dict) -> bool:
    """檢查答案是否與預期答案完全相符。"""
    # ⚠️ 踩雷：outputs / reference_outputs 這兩個參數名不能亂改。
    #    LangSmith 是靠「參數名」決定要注入什麼東西給你的——
    #    看到參數叫 outputs 就塞 target 函式的回傳值，叫 reference_outputs 就塞題庫的標準答案。
    #    這是隱形契約：改成 result / expected 之類的名字，不會有貼心的錯誤訊息提醒你。
    #
    # 分類就 ignore / notify / respond 三選一，有標準答案 → 就該像這樣硬比對。
    # 這種地方動用 LLM 裁判是純浪費錢，還多引入一個會判錯的環節。
    # .lower() 只是防大小寫差異，不是在放水。
    return outputs["classification_decision"].lower() == reference_outputs["classification"].lower()

### 執行 Evaluation

現在的問題是：這些東西是怎麼串接在一起的？evaluate API 會幫我們處理好這件事。它會把 dataset 中的 `inputs` dict 傳給 target 函式；把 dataset 中的 `reference_outputs` dict 傳給 evaluator 函式；並把 agent 的 `outputs` 傳給 evaluator 函式。

請注意，這與我們先前用 Pytest 所做的類似：在 Pytest 中，我們用 `@pytest.mark.parametrize` 把 dataset 範例的 inputs 與 reference outputs 傳入測試函式。

In [9]:
# ⚠️ 這不是程式邏輯，是省錢開關。
# 跑一次 evaluation = 題庫有幾筆就打幾次 LLM，要花錢、也要等。
# 教學 notebook 的常見手法：用一個旗標控制要不要真的跑，
# 之後重讀 notebook 時設成 False，就不會手滑又燒一次錢。
# 若想啟動 evaluation，將其設為 true
run_expt = True
if run_expt:
    # evaluate() 把三個角色湊在一起。這三個分清楚，這章就懂一半了：
    #
    #   target（受測系統）= target_email_assistant：吃 inputs、吐 outputs，就是「考生」。
    #   data（題庫）      = dataset_name：每筆含 inputs（考題）+ reference_outputs（標準答案）。
    #   evaluators（裁判）= classification_evaluator：比對考生答案 vs 標準答案，判對錯。
    #
    # 流程：evaluate 逐筆把 inputs 餵給 target → 收下 outputs
    #      → 連同該筆的 reference_outputs 一起交給 evaluator → 記成績、算統計。
    # 迴圈、通過率、上傳結果都不用自己寫。這就是它跟 Pytest 的分工差異：
    # Pytest 適合「每個案例各自客製檢查」，evaluate 適合「同一個 evaluator 套整份題庫看統計」。
    experiment_results_workflow = client.evaluate(
        # target：受測的系統（考生）
        target_email_assistant,
        # Dataset 名稱
        data=dataset_name,
        # evaluators：裁判函式，可以放多個一次評不同面向
        evaluators=[classification_evaluator],
        # 實驗名稱的前綴，LangSmith 會自動接一段 hash 湊成完整實驗名
        experiment_prefix="E-mail assistant workflow", 
        # 同時進行的 evaluation 數量。
        # 不是不能開大，是開大會撞 OpenAI 的 rate limit，整批實驗紅一片。2 是保守值。
        max_concurrency=2, 
    )

View the evaluation results for experiment: 'E-mail assistant workflow-9f033fa4' at:
https://smith.langchain.com/o/06b59464-6a87-5ed5-a56f-7601ad8dd8e3/datasets/833fb45a-2786-41f4-96b9-fb3f850db407/compare?selectedSessions=89fcb631-308f-49f5-b6d3-f27138989cf6




0it [00:00, ?it/s]

🚫 Classification: IGNORE - This email can be safely ignored
🔔 Classification: NOTIFY - This email contains important information
📧 Classification: RESPOND - This email requires a response
🚫 Classification: IGNORE - This email can be safely ignored
📧 Classification: RESPOND - This email requires a response
🔔 Classification: NOTIFY - This email contains important information
📧 Classification: RESPOND - This email requires a response
🔔 Classification: NOTIFY - This email contains important information
🔔 Classification: NOTIFY - This email contains important information
📧 Classification: RESPOND - This email requires a response
📧 Classification: RESPOND - This email requires a response
🚫 Classification: IGNORE - This email can be safely ignored
📧 Classification: RESPOND - This email requires a response
🔔 Classification: NOTIFY - This email contains important information
📧 Classification: RESPOND - This email requires a response
📧 Classification: RESPOND - This email requires a response


我們可以在 LangSmith UI 中檢視這兩個實驗的結果。

![Test Results](img/eval.png)

## LLM-as-Judge Evaluation

我們已經示範了針對 triage 步驟的單元測試（使用 evaluate()）以及針對 tool calling 的測試（使用 Pytest）。

接下來，我們會示範如何用 LLM 作為評審（judge），依一組成功條件來評估我們 agent 的執行結果。

![types](img/eval_types.png)

首先，我們為 LLM grader 定義一個 structured output schema，其中包含一個評分（grade）以及對該評分的理由說明（justification）。

In [10]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class CriteriaGrade(BaseModel):
    """依特定 criteria 為回應評分。"""
    # 欄位順序是有意義的，不是隨便排的。
    # structured output 的本質，是叫 LLM 照這個 schema 的順序把 JSON 一個 token 一個 token 生出來。
    # justification 放在前面 = 它得先把理由「寫出來」，grade 才輪到生成，
    # 而後生成的 token 讀得到前面已生成的內容——等於逼裁判先想過再判。
    # 反過來 grade 在前，就變成先脫口而出一個答案，再回頭幫自己編理由（跟人一模一樣）。
    #
    # ⚠️ 原始教材在這裡自相矛盾：本 notebook 最後「在更大的測試套件上執行」那段 markdown，
    #    以及 tests/test_response.py 第 28 行，用的都是 grade 在前、justification 在後，跟這裡相反。
    #    這個 cell 的順序才是比較好的做法。看到兩邊不一致，不用懷疑自己。
    justification: str = Field(description="The justification for the grade and score, including specific examples from the response.")
    grade: bool = Field(description="Does the response meet the provided criteria?")
    
# 建立一個全域的 evaluation LLM，避免每個測試都重新建立
#
# ⚠️ 值得點名：裁判用的是 gpt-4o，但受測的 email assistant 用的是 gpt-4.1
#    （見 src/email_assistant/email_assistant.py:21）。裁判和考生不是同一個 model。
#    為什麼要這樣：同一個 model 傾向覺得自己寫的東西不錯，等於自己改自己的考卷。
#    但別高估這招——兩個都是 OpenAI 同家族的 model，只能算部分緩解，不是根治。
#    真要嚴謹，裁判該換一家（例如換 Claude 來評 GPT 寫的信）。
criteria_eval_llm = init_chat_model("openai:gpt-4o")
# with_structured_output = 逼 LLM 照 CriteriaGrade 的 schema 吐 JSON，而不是吐一段散文。
# 這步是 LLM-as-judge 能自動化的關鍵：有了固定 schema，評分結果才是能 assert、能統計的欄位，
# 不用自己去 parse「嗯我覺得這封信寫得還不錯啦」這種鬼東西。
criteria_eval_structured_llm = criteria_eval_llm.with_structured_output(CriteriaGrade)

In [11]:
email_input = email_inputs[0]
print("Email Input:", email_input)
success_criteria = response_criteria_list[0]
print("Success Criteria:", success_criteria)

Email Input: {'author': 'Alice Smith <alice.smith@company.com>', 'to': 'Lance Martin <lance@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': "Hi Lance,\n\nI was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?\n\nSpecifically, I'm looking at:\n- /auth/refresh\n- /auth/validate\n\nThanks!\nAlice"}
Success Criteria: 
• Send email with write_email tool call to acknowledge the question and confirm it will be investigated  



我們的 Email Assistant 會以這封 email 作為輸入被呼叫，而其回應會被格式化成一個字串。接著這些內容全都會被傳給 LLM grader，以取得一個評分以及對該評分的理由說明。

In [12]:
response = email_assistant.invoke({"email_input": email_input})

📧 Classification: RESPOND - This email requires a response


In [13]:
from email_assistant.eval.prompts import RESPONSE_CRITERIA_SYSTEM_PROMPT

# 注意餵給裁判的是「整串 messages」，不是只有最後那封回信。
# 因為 criteria 要問的往往是過程，例如「有沒有先查行事曆才敲會議時間」——
# 只看最後成品，你看不出它是查過才寫，還是憑空瞎掰一個時間、剛好寫得很順。
# format_messages_string 把整段對話（含 tool calls）壓成一個字串，
# 因為裁判 LLM 只吃文字，不吃 message 物件。
all_messages_str = format_messages_string(response['messages'])
# 這就是 LLM-as-judge 的全貌，拆開看其實很樸素，沒有魔法：
#   system prompt = 給裁判的評分守則（下一個 cell 會把它印出來看）
#   user message  = 「這是評分標準 + 這是考生的作答，請評分並說明理由」
# 就是再打一次 LLM，只是這次它的工作是評分，不是回信。
eval_result = criteria_eval_structured_llm.invoke([
        {"role": "system",
            "content": RESPONSE_CRITERIA_SYSTEM_PROMPT},
        {"role": "user",
            "content": f"""\n\n Response criteria: {success_criteria} \n\n Assistant's response: \n\n {all_messages_str} \n\n Evaluate whether the assistant's response meets the criteria and provide justification for your evaluation."""}
    ])

eval_result

CriteriaGrade(justification='The assistant\'s response meets the provided criteria because:\n\n1. **Acknowledgement and Confirmation**: The email acknowledges the question asked by Alice Smith and confirms that it will be investigated. This is evident in the response content: "Thank you for pointing this out. I\'ll investigate whether the /auth/refresh and /auth/validate endpoints were intentionally omitted from the API documentation or if an update is needed."\n\n2. **Use of the write_email Tool**: The write_email tool was correctly used to send the email, as shown in the tool call logs.\n\nBoth criteria specified have been satisfied by the response.', grade=True)

In [14]:
RESPONSE_CRITERIA_SYSTEM_PROMPT

"You are evaluating an email assistant that works on behalf of a user.\n\nYou will see a sequence of messages, starting with an email sent to the user. \n\nYou will then see the assistant's response to this email on behalf of the user, which includes any tool calls made (e.g., write_email, schedule_meeting, check_calendar_availability, done).\n\nYou will also see a list of criteria that the assistant's response must meet.\n\nYour job is to evaluate if the assistant's response meets ALL the criteria bullet points provided.\n\nIMPORTANT EVALUATION INSTRUCTIONS:\n1. The assistant's response is formatted as a list of messages.\n2. The response criteria are formatted as bullet points (•)\n3. You must evaluate the response against EACH bullet point individually\n4. ALL bullet points must be met for the response to receive a 'True' grade\n5. For each bullet point, cite specific text from the response that satisfies or fails to satisfy it\n6. Be objective and rigorous in your evaluation\n7. In

我們可以看到，LLM grader 回傳的 eval result 具有與我們 `CriteriaGrade` base model 相符的 schema。

## 在更大的測試套件上執行
現在我們已經看過如何用 Pytest 與 evaluate() 來評估 agent，也看過一個用 LLM 作為評審的範例，接著我們可以在一個更大的測試套件上執行 evaluation，以便更全面地了解 agent 在更多樣化範例上的表現。

我們來讓 email_assistant 對一個更大的測試套件執行。
```
! LANGSMITH_TEST_SUITE='Email assistant: Test Full Response Interrupt' LANGSMITH_EXPERIMENT='email_assistant' pytest tests/test_response.py --agent-module email_assistant
```

在 `test_response.py` 中，你可以看到幾件事。

我們把 dataset 範例傳入會執行 pytest 並記錄到我們 `LANGSMITH_TEST_SUITE` 的函式中：

```
# Reference output key
@pytest.mark.langsmith(output_keys=["criteria"])
# Variable names and a list of tuples with the test cases
# Each test case is (email_input, email_name, criteria, expected_calls)
@pytest.mark.parametrize("email_input,email_name,criteria,expected_calls",create_response_test_cases())
def test_response_criteria_evaluation(email_input, email_name, criteria, expected_calls):
```

我們搭配一個評分用的 schema 使用 LLM-as-judge：
```
class CriteriaGrade(BaseModel):
    """Score the response against specific criteria."""
    grade: bool = Field(description="Does the response meet the provided criteria?")
    justification: str = Field(description="The justification for the grade and score, including specific examples from the response.")
```


我們依據 criteria 來評估 agent 的回應：
```
    # Evaluate against criteria
    eval_result = criteria_eval_structured_llm.invoke([
        {"role": "system",
            "content": RESPONSE_CRITERIA_SYSTEM_PROMPT},
        {"role": "user",
            "content": f"""\n\n Response criteria: {criteria} \n\n Assistant's response: \n\n {all_messages_str} \n\n Evaluate whether the assistant's response meets the criteria and provide justification for your evaluation."""}
    ])
```

In [16]:
# 上面那段指令是寫在 markdown 裡的純文字，複製出來直接跑會踩兩個坑。
# 這一格是修好的、真的按得下去的版本。
#
# 坑 1：教材沒提 LANGSMITH_TRACING=true
#   少了它，@pytest.mark.langsmith 的 log_inputs 會直接拋
#   ValueError: log_inputs should only be called ... with tracing enabled。
#   .env.example 和 README 都有寫要設，但 cp 出來的 .env 很容易漏掉。
#   （已補進 ../.env，所以這行指令不用再自己帶。）
#
# 坑 2：! 指令的工作目錄是 notebooks/，但 tests/ 在上一層
#   直接 `pytest tests/test_response.py` 一定找不到檔案 → 所以要先 cd ..
#
# ⚠️ 這一格會真的花錢：16 個 case，每個 case 各一次 agent 執行 + 一次 LLM-as-judge 評分，
#    實測約 80 秒跑完（13 passed / 1 failed）。這就是「改 prompt 前後各跑一次」的真實成本。
#
# --agent-module 是這個專案自訂的 pytest 參數（定義在 tests/conftest.py），
# 用來切換要測哪一版 assistant：email_assistant / email_assistant_hitl / email_assistant_hitl_memory。
!cd .. && LANGSMITH_TEST_SUITE='Email assistant: Test Full Response Interrupt' LANGSMITH_EXPERIMENT='email_assistant' pytest tests/test_response.py --agent-module email_assistant

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0
rootdir: /Users/kevinluo/langgraph-deepagents-book-make/docs/langchain-academy/langgraph/langgraph-atonomous-agent/agents-from-scratch
configfile: pyproject.toml
plugins: anyio-4.14.1, xdist-3.8.0, langsmith-0.9.3
collected 16 items                                                             

tests/test_response.py .........F..F...                                  [100%]

=================================== FAILURES ===================================
_ test_response_criteria_evaluation[email_input1-email_input_4-\n\u2022 Check calendar availability for Tuesday or Thursday afternoon next week with check_calendar_availability tool call \n\u2022 Confirm availability for a 45-minute meeting\n\u2022 Send calendar invite with schedule_meeting tool call \n\u2022 Send email with write_email tool call to acknowledge tax planning request and notifying t

現在，我們來在 LangSmith UI 中檢視這個實驗，看看我們的 agent 哪裡做得好、哪裡還有改進空間。

#### 取得結果

我們也可以透過讀取與實驗相關聯的 tracing 專案來取得 evaluation 的結果。這對於替我們 agent 的表現製作自訂的視覺化呈現非常有用。

In [15]:
# ⚠️ 踩雷：這是「別人跑出來的」實驗名稱，照著跑一定失敗——你的 LangSmith 專案裡沒這東西。
#    正確做法：先跑上一個 cell 那行 pytest 指令（tests/test_response.py），
#    跑完去 LangSmith UI 的 Datasets & Experiments 找到實際產生的實驗名稱，貼回這裡。
#    冒號後面那串 8286b3b8 是隨機 hash，你猜不到，也不用猜。
# TODO: 把你的實驗名稱貼到這裡
experiment_name = "email_assistant:8286b3b8"
# 又一個省錢／省時開關（同前面的 run_expt）：預設 False，想讀結果時才打開。
# 設為 True 以載入實驗結果
load_expt = False
if load_expt:
    # 換個角度看評估結果：前面看的是「答對幾題」，這裡看的是「跑起來貴不貴、慢不慢」。
    # 這件事很容易被忽略：一個 agent 準確率再漂亮，p99 延遲 30 秒、每跑一次燒十萬 token，
    # 正式環境還是不能用。read_project 把成本面的數字撈出來，讓架構決策有依據——
    # 例如「換成小 model，準確率掉 3%、成本掉 80%」到底划不划算。
    email_assistant_experiment_results = client.read_project(project_name=experiment_name, include_stats=True)
    print("Latency p50:", email_assistant_experiment_results.latency_p50)
    print("Latency p99:", email_assistant_experiment_results.latency_p99)
    print("Token Usage:", email_assistant_experiment_results.total_tokens)
    print("Feedback Stats:", email_assistant_experiment_results.feedback_stats)